# Lecture 08 — One fitted model, one saved policy, one API

Follow one artifact through fitting, validation-selected threshold, final assessment, saving, loading, and an HTTP request. First complete the core route. MLflow and Docker are optional extensions. No background server is started by this notebook.

![Teaching figure for deployment](../../docs/figures/08_deployment.svg)

*Conceptual serving workflow. The same saved preprocessing is applied to each validated incoming record.*



## 1. What crosses the boundary?

An **artifact** is a file containing the fitted preprocessing, predictor, and metadata. A server **process** loads it into memory. An **API** is an interface another program calls. An HTTP **request** sends a JSON object (named fields and values) to an **endpoint**, a URL path such as `/predict`. A **response** returns data with a status code: 200 for success, 422 for invalid fields in this API, and 503 if its model cannot load.

Our chain is raw fields → validation → saved preprocessing → probability → stored threshold → JSON. Neither scaling nor model coefficients are refitted when a request arrives. We use an in-process client to exercise the same endpoint without managing a live server.

In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import platform
import joblib
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, roc_auc_score

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data/churn.csv").is_file())
sys.path.insert(0, str(ROOT / "src"))
from course_ml.train import build_pipeline, FEATURES, CONTRACTS
from course_ml.teaching import local_prediction_client

DATA = ROOT / "data/churn.csv"
OUTPUT = ROOT / ".verification" / "deployment-core"
OUTPUT.mkdir(parents=True, exist_ok=True)
# A unique filename avoids serving a cached artifact during notebook reruns.
from uuid import uuid4
artifact_path = OUTPUT / f"churn-{uuid4().hex}.joblib"
data = pd.read_csv(DATA)
print("Rows:", len(data))
display(data.head(3))

## 2. Fit once, choose a policy on validation

Hold aside 25% for final assessment. Split development rows again for fitting and validation. The pipeline learns standardization, category encoding, and logistic coefficients only on fitting rows. We assume illustrative costs: a missed churn costs five units and an unnecessary contact costs one unit. Select the cutoff minimizing validation cost; equal minima choose the smallest cutoff. Keep both the fitted pipeline and this cutoff fixed afterward. We do not refit on validation rows, which would change the probabilities underlying the selected policy.

In [ ]:
X, y = data[FEATURES], data["churn"].astype(int)
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_development, y_development, test_size=0.20, random_state=17, stratify=y_development
)
pipeline = build_pipeline().fit(X_fit, y_fit)
positive_column = list(pipeline.classes_).index(1)
validation_probabilities = pipeline.predict_proba(X_validation)[:, positive_column]
C_FN, C_FP = 5.0, 1.0
validation_rows = []
for cutoff in np.linspace(0.05, 0.95, 19):
    decisions = validation_probabilities >= cutoff
    fn = np.sum((y_validation.to_numpy() == 1) & ~decisions)
    fp = np.sum((y_validation.to_numpy() == 0) & decisions)
    validation_rows.append({"threshold": float(cutoff), "cost": C_FN * fn + C_FP * fp,
                            "alerts": int(decisions.sum())})
validation_table = pd.DataFrame(validation_rows)
selected = validation_table.sort_values(["cost", "threshold"]).iloc[0]
threshold = float(selected.threshold)
print("Validation-selected threshold:", threshold)
display(validation_table)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(validation_table.threshold, validation_table.cost, "o-")
ax.scatter([threshold], [selected.cost], color="crimson", zorder=3, label="Selected on validation")
ax.set(xlabel="Threshold", ylabel="Illustrative validation cost", title="Choose the policy before final assessment")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Assess the frozen pipeline and save the same policy

Use the final test once to describe performance, not to revise the cutoff. Save that exact pipeline and threshold together. Metadata records the schema, source-data hash, dependency versions, and partition sizes. `joblib` stores Python objects; load only artifacts whose source you trust, and reproduce their environment. This file is the single artifact used throughout the remaining core lesson.

In [ ]:
test_probabilities = pipeline.predict_proba(X_test)[:, positive_column]
test_decisions = test_probabilities >= threshold
metrics = {
    "roc_auc": float(roc_auc_score(y_test, test_probabilities)),
    "precision": float(precision_score(y_test, test_decisions, zero_division=0)),
    "recall": float(recall_score(y_test, test_decisions, zero_division=0)),
    "cost": float(C_FN * np.sum((y_test.to_numpy() == 1) & ~test_decisions)
                  + C_FP * np.sum((y_test.to_numpy() == 0) & test_decisions)),
}
metadata = {
    "artifact_format": 1, "features": FEATURES, "contract_values": CONTRACTS,
    "positive_class": 1, "threshold": threshold, "metrics": metrics,
    "fit_rows": len(X_fit), "validation_rows": len(X_validation), "test_rows": len(X_test),
    "data_sha256": hashlib.sha256(DATA.read_bytes()).hexdigest(),
    "dependency_versions": {"python": platform.python_version(), "scikit_learn": sklearn.__version__,
                            "pandas": pd.__version__, "numpy": np.__version__},
}
artifact = {"pipeline": pipeline, "metadata": metadata}
joblib.dump(artifact, artifact_path)
print("Saved:", artifact_path.name)
display(pd.Series(metrics, name="Final test metrics"))

## 4. A successful prediction before and after saving

This fictional customer is a software-contract probe, not another evaluation dataset. Compare the probability from the in-memory model with the reloaded one, then check that the stored threshold produces the same decision. Saving parameters should not change the function.

In [ ]:
customer = {"tenure_months": 8, "monthly_charge": 92.0,
            "support_calls": 4, "contract": "monthly"}
customer_row = pd.DataFrame([customer], columns=FEATURES)
before_saving = float(pipeline.predict_proba(customer_row)[0, positive_column])
loaded = joblib.load(artifact_path)
after_loading = float(loaded["pipeline"].predict_proba(customer_row)[0, positive_column])
np.testing.assert_allclose(before_saving, after_loading, rtol=0, atol=1e-12)
print({"before_saving": before_saving, "after_loading": after_loading,
       "threshold": loaded["metadata"]["threshold"],
       "decision": int(after_loading >= loaded["metadata"]["threshold"])})

## 5. Send the same customer to the API

The helper below points a local test client at our saved artifact. `client.post` sends the record to the actual FastAPI endpoint in this process. `.json()` decodes the response. The assertions verify probability, threshold, and decision parity across all three paths. The helper's environment restoration is explained in the appendix.

In [ ]:
with local_prediction_client(artifact_path) as client:
    response = client.post("/predict", json=customer)
    assert response.status_code == 200
    result = response.json()
np.testing.assert_allclose(result["churn_probability"], before_saving, rtol=0, atol=1e-12)
assert result["threshold"] == threshold
assert result["prediction"] == int(before_saving >= threshold)
print("HTTP status:", response.status_code)
print(json.dumps(result, indent=2))
print("Parity verified: before saving = after loading = API probability.")

## 6. Reject an invalid request

A contract value of `quarterly` is outside the documented schema. The API rejects this input before scoring it. A working statistical model and a working request contract are separate obligations.

In [ ]:
with local_prediction_client(artifact_path) as client:
    invalid = client.post("/predict", json={**customer, "contract": "quarterly"})
assert invalid.status_code == 422
print("Invalid request status:", invalid.status_code)
print(json.dumps(invalid.json(), indent=2))

## 7. Inspect the contract you will hand off

The artifact on disk is loaded into memory by a process. Changing a file is not the same as retraining; a long-running process may cache the old model and require a controlled reload. Input drift can invalidate statistical assumptions even while every HTTP request succeeds. API correctness does not guarantee calibration or real-world usefulness.

The synthetic-data example is a local teaching service. Authentication, TLS, rate limits, privacy controls, and operational monitoring are not implemented. Use fictional records and keep the service local.

In [ ]:
display(pd.Series(loaded["metadata"]))
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(["Before saving", "After loading", "API"],
       [before_saving, after_loading, result["churn_probability"]], color=["#285b85", "#26938a", "#c47739"])
ax.axhline(threshold, color="black", linestyle="--", label="Saved decision threshold")
ax.set(ylim=(0, 1), ylabel="Churn probability", title="One artifact, three equivalent inference paths")
ax.legend()
plt.tight_layout()
plt.show()

## Optional extension: track this exact artifact with MLflow

Tracking records what ran; it does not train a better model. This extension logs the same artifact, metrics, cutoff, and data hash without fitting another estimator. The local tracking URI is confined to the MLflow client object, so no global environment changes are needed.

In [ ]:
from mlflow import MlflowClient
tracking_directory = ROOT / ".verification" / "deployment-tracking"
tracking_directory.mkdir(parents=True, exist_ok=True)
tracking_client = MlflowClient(tracking_uri="sqlite:///" + (tracking_directory / "mlflow.db").as_posix())
experiment_name = "single-artifact-deployment"
experiment = tracking_client.get_experiment_by_name(experiment_name)
experiment_id = (experiment.experiment_id if experiment else
                 tracking_client.create_experiment(experiment_name))
run = tracking_client.create_run(experiment_id)
run_id = run.info.run_id
tracking_client.log_param(run_id, "threshold", threshold)
tracking_client.set_tag(run_id, "data_sha256", metadata["data_sha256"])
for name, value in metrics.items():
    tracking_client.log_metric(run_id, name, value)
tracking_client.log_artifact(run_id, str(artifact_path))
tracking_client.set_terminated(run_id)
print("Tracked the same saved artifact in run:", run_id)

## Optional extension: a separate server process and Docker

The README gives commands for starting Uvicorn and building the Docker image. The command-line trainer there produces its own independently trained artifact with its documented default threshold; it does not automatically use the notebook artifact. To serve this notebook's policy, set `MODEL_PATH` to the saved path printed above before starting `python -m uvicorn course_ml.app:app --host 127.0.0.1 --port 8001`. Do not confuse that process with the website server on port 8000.

## Appendix: what the helper hides

`course_ml.teaching.local_prediction_client` temporarily sets `MODEL_PATH`, clears the API's artifact cache, opens `TestClient`, then restores environment and cache in a `finally` block. It is for sequential notebook demonstrations, not concurrent server routing. Inspect `src/course_ml/teaching.py` after you understand the core workflow. Artifact files remain under ignored `.verification/deployment-core` so you can inspect them. Only remove them after closing processes that use them.

## Check your understanding

Which state was learned on fitting rows? Which choice used validation? Why did we keep the pipeline fixed afterward? What would have to match for a colleague to reproduce the saved function? Explain why identical responses after saving are a software guarantee, while prediction quality on new customers is a statistical question.